In [1]:
import itertools
from collections import defaultdict

import numpy as np
from qiskit.circuit import ParameterVector, QuantumCircuit

# Qiskit Circuits
from qiskit.circuit.library import TwoLocal
from qiskit.primitives import StatevectorSampler
from qiskit_aer.primitives import Sampler as AerSampler

# Qiskit Algorithms & Primitives
from qiskit_algorithms.minimum_eigensolvers import (
    QAOA,
    NumPyMinimumEigensolver,
    SamplingVQE,
)
from qiskit_algorithms.optimizers import COBYLA

# Qiskit Optimization
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.algorithms import MinimumEigenOptimizer

# ============================================================
# 1. PROBLEM SETUP
# ============================================================
dist = np.array(
    [
        [0, 3.9, 6, 5],
        [4, 0, 3, 2],
        [6, 3, 0, 4],
        [5, 2, 4, 0],
    ],
    dtype=float,
)

scenic = {1: 5.0, 2: 9.0, 3: 6.0}
pois = [1, 2, 3]  # points of interest
slots = [1, 2]  # time slots
A, B = 12.0, 10.0  # penalty coefficients

# ============================================================
# 2. BUILD QUBO
# ============================================================
qp = QuadraticProgram("toy_hiking_route")
for i in pois:
    for p in slots:
        qp.binary_var(name=f"x_{i}_{p}")

linear, quadratic = defaultdict(float), defaultdict(float)
constant = 0.0

for i in pois:
    linear[f"x_{i}_1"] += dist[0][i] - scenic[i]
    linear[f"x_{i}_2"] += dist[i][0] - scenic[i]

for i in pois:
    for j in pois:
        quadratic[(f"x_{i}_1", f"x_{j}_2")] += dist[i][j]

for p in slots:
    constant += A
    for i in pois:
        linear[f"x_{i}_{p}"] += -A
    for i, j in itertools.combinations(pois, 2):
        quadratic[(f"x_{i}_{p}", f"x_{j}_{p}")] += 2 * A

for i in pois:
    quadratic[(f"x_{i}_1", f"x_{i}_2")] += B

qp.minimize(constant=constant, linear=dict(linear), quadratic=dict(quadratic))
op, offset = qp.to_ising()


# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================
def decode_solution(xvec):
    var_names = [f"x_{i}_{p}" for i in pois for p in slots]
    sol = {name: int(round(val)) for name, val in zip(var_names, xvec)}
    first, second = None, None
    for i in pois:
        if sol[f"x_{i}_1"] == 1:
            first = i
        if sol[f"x_{i}_2"] == 1:
            second = i
    return [0, first, second, 0]


def create_hea_ansatz(qubits, layers):
    qc = QuantumCircuit(qubits)
    thetas = ParameterVector("theta", qubits * layers)

    for i in range(layers):
        # Rotation Layer
        for j in range(qubits):
            qc.ry(thetas[i * qubits + j], j)

        # Entanglement Layer (Ring topology)
        for j in range(qubits - 1):
            qc.cx(j, j + 1)
        if qubits > 1:
            qc.cx(qubits - 1, 0)  # Close the ring

        if layers > 1:
            qc.barrier()

    return qc


# ============================================================
# 4. SOLVERS
# ============================================================
sampler = StatevectorSampler()

# --- Exact ---
exact = MinimumEigenOptimizer(NumPyMinimumEigensolver()).solve(qp)

# --- QAOA ---
qaoa = MinimumEigenOptimizer(
    QAOA(sampler=sampler, optimizer=COBYLA(maxiter=300), reps=2)
).solve(qp)

# --- Standard VQE (TwoLocal) ---
ansatz_twolocal = TwoLocal(
    num_qubits=op.num_qubits,
    rotation_blocks=["ry", "rz"],
    entanglement_blocks="cz",
    reps=3,
)
vqe = MinimumEigenOptimizer(
    SamplingVQE(sampler=sampler, ansatz=ansatz_twolocal, optimizer=COBYLA(maxiter=800))
).solve(qp)

# --- Custom HEA VQE ---
custom_ansatz = create_hea_ansatz(op.num_qubits, layers=3)
custom_vqe_solver = SamplingVQE(
    sampler=sampler, ansatz=custom_ansatz, optimizer=COBYLA(maxiter=800)
)
custom_vqe_result = MinimumEigenOptimizer(custom_vqe_solver).solve(qp)

# ============================================================
# 5. OUTPUT
# ============================================================
print(f"\n{'Solver':<16} | {'Route':<15} | {'Cost':<10}")
print("-" * 47)
print(f"{'EXACT':<16} | {str(decode_solution(exact.x)):<15} | {exact.fval:<10.2f}")
print(f"{'QAOA':<16} | {str(decode_solution(qaoa.x)):<15} | {qaoa.fval:<10.2f}")
print(f"{'Standard VQE':<16} | {str(decode_solution(vqe.x)):<15} | {vqe.fval:<10.2f}")
print(
    f"{'Custom HEA VQE':<16} | {str(decode_solution(custom_vqe_result.x)):<15} | {custom_vqe_result.fval:<10.2f}"
)

/var/folders/1t/jzfpxh1x21d2665w__9wptgc0000gp/T/ipykernel_34222/1706439237.py:110: DeprecationWarning: The class ``qiskit.circuit.library.n_local.two_local.TwoLocal`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.n_local instead.
  ansatz_twolocal = TwoLocal(num_qubits=op.num_qubits, rotation_blocks=["ry", "rz"], entanglement_blocks="cz", reps=3)
/Users/stabilini/Desktop/Benasque_hack/.venv/lib/python3.13/site-packages/scipy/_lib/pyprima/common/linalg.py:61: RuntimeWarning: divide by zero encountered in matmul
  return x@y
/Users/stabilini/Desktop/Benasque_hack/.venv/lib/python3.13/site-packages/scipy/_lib/pyprima/common/linalg.py:61: RuntimeWarning: overflow encountered in matmul
  return x@y
/Users/stabilini/Desktop/Benasque_hack/.venv/lib/python3.13/site-packages/scipy/_lib/pyprima/common/linalg.py:61: RuntimeWarning: invalid value encountered in matmul
  return x@y
/Users/stabilini/Desktop/Benasque_hack/.venv/lib/python3.


Solver           | Route           | Cost      
-----------------------------------------------
EXACT            | [0, 1, 2, 0]    | -1.10     
QAOA             | [0, 1, 2, 0]    | -1.10     
Standard VQE     | [0, 1, 2, 0]    | -1.10     
Custom HEA VQE   | [0, 1, 2, 0]    | -1.10     


In [2]:
import itertools
from collections import defaultdict

import numpy as np

# --- NEW: Tensor Network Integration ---
# Qiskit Circuits
from qiskit.circuit.library import TwoLocal
from qiskit.primitives import StatevectorSampler

# Qiskit Algorithms & Primitives
from qiskit_algorithms.minimum_eigensolvers import (
    QAOA,
    NumPyMinimumEigensolver,
    SamplingVQE,
)
from qiskit_algorithms.optimizers import COBYLA

# Qiskit Optimization
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.algorithms import MinimumEigenOptimizer

# ============================================================
# 1. ACTUAL DATA SETUP (Extracted from Hackathon CSV)
# ============================================================
# 0: Benasque, 1: Cerler, 2: Ancils, 3: Portillón de Benás
dist = np.array(
    [
        [0.00, 1.20, 0.45, 2.50],  # From Benasque
        [1.20, 0.00, 1.30, 3.00],  # From Cerler
        [0.45, 1.30, 0.00, 1.42],  # From Ancils (1.42h to Portillon in CSV)
        [2.50, 3.00, 1.42, 0.00],  # From Portillón
    ],
    dtype=float,
)

# Scenic values based on Elevation (Normalized to toy scale)
# Cerler (1530m), Ancils (1140m), Portillón (2444m)
scenic = {1: 6.51, 2: 5.02, 3: 10.0}

pois = [1, 2, 3]  # points of interest
slots = [1, 2]  # time slots
A, B = 15.0, 12.0  # Penalty coefficients (Slightly higher for real distance scale)

# ============================================================
# 2. BUILD QUBO
# ============================================================
qp = QuadraticProgram("benasque_hiking_route")
for i in pois:
    for p in slots:
        qp.binary_var(name=f"x_{i}_{p}")

linear, quadratic = defaultdict(float), defaultdict(float)
constant = 0.0

for i in pois:
    linear[f"x_{i}_1"] += dist[0][i] - scenic[i]
    linear[f"x_{i}_2"] += dist[i][0] - scenic[i]

for i in pois:
    for j in pois:
        quadratic[(f"x_{i}_1", f"x_{j}_2")] += dist[i][j]

for p in slots:
    constant += A
    for i in pois:
        linear[f"x_{i}_{p}"] += -A
    for i, j in itertools.combinations(pois, 2):
        quadratic[(f"x_{i}_{p}", f"x_{j}_{p}")] += 2 * A

for i in pois:
    quadratic[(f"x_{i}_1", f"x_{i}_2")] += B

qp.minimize(constant=constant, linear=dict(linear), quadratic=dict(quadratic))
op, offset = qp.to_ising()


# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================
def decode_solution(xvec):
    var_names = [f"x_{i}_{p}" for i in pois for p in slots]
    sol = {name: int(round(val)) for name, val in zip(var_names, xvec)}
    first, second = None, None
    for i in pois:
        if sol[f"x_{i}_1"] == 1:
            first = i
        if sol[f"x_{i}_2"] == 1:
            second = i
    return [0, first, second, 0]


def create_hea_ansatz(qubits, layers):
    qc = QuantumCircuit(qubits)
    thetas = ParameterVector("theta", qubits * layers)
    for i in range(layers):
        for j in range(qubits):
            qc.ry(thetas[i * qubits + j], j)
        for j in range(qubits - 1):
            qc.cx(j, j + 1)
        if qubits > 1:
            qc.cx(qubits - 1, 0)
        if layers > 1:
            qc.barrier()
    return qc


# ============================================================
# 4. SOLVERS (Now with Tensor Network Integration)
# ============================================================
sampler = StatevectorSampler()

# --- TENSOR NETWORK Integration (MPS Method) ---
# This uses the Matrix Product State simulator for scalability
tn_sampler = AerSampler(
    backend_options={
        "method": "matrix_product_state",
        "matrix_product_state_max_bond_dimension": 16,
        "device": "CPU",
    }
)

# --- Exact ---
exact = MinimumEigenOptimizer(NumPyMinimumEigensolver()).solve(qp)

# --- QAOA ---
qaoa = MinimumEigenOptimizer(
    QAOA(sampler=sampler, optimizer=COBYLA(maxiter=300), reps=3)
).solve(qp)

# --- Standard VQE ---
ansatz_twolocal = TwoLocal(
    num_qubits=op.num_qubits,
    rotation_blocks=["ry", "rz"],
    entanglement_blocks="cz",
    reps=3,
)
vqe = MinimumEigenOptimizer(
    SamplingVQE(sampler=sampler, ansatz=ansatz_twolocal, optimizer=COBYLA(maxiter=800))
).solve(qp)

# --- Custom HEA VQE ---
custom_ansatz = create_hea_ansatz(op.num_qubits, layers=4)
custom_vqe_solver = SamplingVQE(
    sampler=sampler, ansatz=custom_ansatz, optimizer=COBYLA(maxiter=800)
)
custom_vqe_result = MinimumEigenOptimizer(custom_vqe_solver).solve(qp)


# # --- Custom HEA VQE using TENSOR NETWORKS (MPS) ---
# custom_ansatz = create_hea_ansatz(op.num_qubits, layers=3)
# # We swap the sampler to tn_sampler here!
# custom_vqe_solver = SamplingVQE(sampler=tn_sampler, ansatz=custom_ansatz, optimizer=COBYLA(maxiter=800))
# custom_vqe_result = MinimumEigenOptimizer(custom_vqe_solver).solve(qp)

# ============================================================
# 5. OUTPUT
# ============================================================
print(f"\n{'Solver':<18} | {'Route':<15} | {'Cost':<10}")
print("-" * 50)
print(f"{'EXACT':<18} | {str(decode_solution(exact.x)):<15} | {exact.fval:<10.2f}")
print(f"{'QAOA':<18} | {str(decode_solution(qaoa.x)):<15} | {qaoa.fval:<10.2f}")
print(f"{'Standard VQE':<18} | {str(decode_solution(vqe.x)):<15} | {vqe.fval:<10.2f}")
print(
    f"{'Custom HEA VQE':<18} | {str(decode_solution(custom_vqe_result.x)):<15} | {custom_vqe_result.fval:<10.2f}"
)
# print(f"{'Tensor Network VQE':<18} | {str(decode_solution(custom_vqe_result.x)):<15} | {custom_vqe_result.fval:<10.2f}")

/var/folders/1t/jzfpxh1x21d2665w__9wptgc0000gp/T/ipykernel_34222/3686240896.py:118: DeprecationWarning: The class ``qiskit.circuit.library.n_local.two_local.TwoLocal`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.n_local instead.
  ansatz_twolocal = TwoLocal(num_qubits=op.num_qubits, rotation_blocks=["ry", "rz"],
/Users/stabilini/Desktop/Benasque_hack/.venv/lib/python3.13/site-packages/scipy/_lib/pyprima/common/linalg.py:61: RuntimeWarning: divide by zero encountered in matmul
  return x@y
/Users/stabilini/Desktop/Benasque_hack/.venv/lib/python3.13/site-packages/scipy/_lib/pyprima/common/linalg.py:61: RuntimeWarning: overflow encountered in matmul
  return x@y
/Users/stabilini/Desktop/Benasque_hack/.venv/lib/python3.13/site-packages/scipy/_lib/pyprima/common/linalg.py:61: RuntimeWarning: invalid value encountered in matmul
  return x@y
/Users/stabilini/Desktop/Benasque_hack/.venv/lib/python3.13/site-packages/scipy/_lib/pyprim


Solver             | Route           | Cost      
--------------------------------------------------
EXACT              | [0, 2, 3, 0]    | -10.65    
QAOA               | [0, 3, 2, 0]    | -10.65    
Standard VQE       | [0, 2, 3, 0]    | -10.65    
Custom HEA VQE     | [0, 3, 2, 0]    | -10.65    


In [ ]:
from tenpy.algorithms import dmrg
from tenpy.models.lattice import Chain
from tenpy.models.model import MPOModel
from tenpy.networks.mpo import MPOGraph
from tenpy.networks.mps import MPS
from tenpy.networks.site import SpinHalfSite
from tenpy.networks.terms import CouplingTerms, OnsiteTerms

In [10]:
def build_mpo_from_ising(sparse_pauli_op, n_qubits):
    """
    Convert a Qiskit Ising Hamiltonian (SparsePauliOp with I/Z strings)
    into a TeNPy MPOModel for finite-size DMRG.
    """

    site = SpinHalfSite(conserve="None")
    sites = [site] * n_qubits

    if "Sigmaz" in site.opnames:
        z_op = "Sigmaz"
        z_scale = 1.0
    else:
        z_op = "Sz"
        z_scale = 2.0

    h = np.zeros(n_qubits, dtype=float)
    J = np.zeros((n_qubits, n_qubits), dtype=float)

    for pauli_str, coeff in zip(
        sparse_pauli_op.paulis.to_labels(), sparse_pauli_op.coeffs
    ):
        coeff = float(np.real_if_close(coeff))
        if abs(coeff) < 1e-14:
            continue

        active = [(i, p) for i, p in enumerate(reversed(pauli_str)) if p != "I"]

        if len(active) == 1:
            i, p = active[0]
            if p != "Z":
                raise ValueError(
                    f"Unsupported 1-body Pauli term '{p}' in Ising Hamiltonian"
                )
            h[i] += coeff
        elif len(active) == 2:
            (i, p_i), (j, p_j) = active
            if p_i != "Z" or p_j != "Z":
                raise ValueError("Unsupported 2-body Pauli term in Ising Hamiltonian")
            i, j = sorted((i, j))
            J[i, j] += coeff
        elif len(active) > 2:
            raise ValueError(
                "Found higher-order term; expected at most quadratic Ising terms"
            )

    onsite = OnsiteTerms(n_qubits)
    for i in range(n_qubits):
        if abs(h[i]) > 1e-12:
            onsite.add_onsite_term(h[i] * z_scale, i, z_op)

    coupling = CouplingTerms(n_qubits)
    for i in range(n_qubits):
        for j in range(i + 1, n_qubits):
            if abs(J[i, j]) > 1e-12:
                coupling.add_coupling_term(J[i, j] * (z_scale**2), i, j, z_op, z_op)

    graph = MPOGraph(sites, bc="finite")
    onsite.add_to_graph(graph)
    coupling.add_to_graph(graph)
    graph.add_missing_IdL_IdR()

    H_mpo = graph.build_MPO()
    lat = Chain(n_qubits, site, bc="open", bc_MPS="finite")
    model = MPOModel(lat, H_mpo)

    return sites, model, z_op

In [11]:
def solve_with_tenpy_dmrg(qp, bond_dim=32, n_sweeps=10):
    op, offset = qp.to_ising()
    n = op.num_qubits

    sites, model, z_op = build_mpo_from_ising(op, n)

    psi0 = MPS.from_product_state(sites, ["up"] * n, bc="finite")

    dmrg_params = {
        "trunc_params": {
            "chi_max": bond_dim,
            "svd_min": 1e-10,
        },
        "max_sweeps": n_sweeps,
        "N_sweeps_check": 1,
        "mixer": None,
        "combine": False,
    }

    engine = dmrg.TwoSiteDMRGEngine(psi0, model, dmrg_params)
    energy, psi_ground = engine.run()

    z_exp = np.asarray(psi_ground.expectation_value(z_op), dtype=float)
    x_tn = (z_exp < 0.0).astype(int)

    return x_tn, float(energy + offset)

In [12]:
x_tn, energy_tn = solve_with_tenpy_dmrg(qp, bond_dim=32, n_sweeps=10)

print(f"\n{'Solver':<16} | {'Route':<15} | {'Cost':<10}")
print("-" * 47)
print(
    f"{'EXACT (NumPy)':<16} | {str(decode_solution(exact.x)):<15} | {exact.fval:<10.2f}"
)
print(f"{'TN (TeNPy)':<16} | {str(decode_solution(x_tn)):<15} | {energy_tn:<10.2f}")
print(f"{'QAOA':<16} | {str(decode_solution(qaoa.x)):<15} | {qaoa.fval:<10.2f}")
print(f"{'Standard VQE':<16} | {str(decode_solution(vqe.x)):<15} | {vqe.fval:<10.2f}")
print(
    f"{'Custom HEA VQE':<16} | {str(decode_solution(custom_vqe_result.x)):<15} | {custom_vqe_result.fval:<10.2f}"
)


Solver           | Route           | Cost      
-----------------------------------------------
EXACT (NumPy)    | [0, 2, 3, 0]    | -10.65    
TN (TeNPy)       | [0, 1, 1, 0]    | 1.38      
QAOA             | [0, 3, 2, 0]    | -10.65    
Standard VQE     | [0, 2, 3, 0]    | -10.65    
Custom HEA VQE   | [0, 3, 2, 0]    | -10.65    


/Users/stabilini/Desktop/Benasque_hack/.venv/lib/python3.13/site-packages/tenpy/networks/mpo.py:2192: UserWarning: unit_cell_width is a new argument for MPS and similar classes. It is optional for now, but will become mandatory in a future release. The default value (unit_cell_width=len(sites)) is correct, iff the lattice is a Chain. For other lattices, it is incorrect. It is used for dipolar charges and correlation_function2.
  super().__init__(sites=sites, bc=bc, unit_cell_width=unit_cell_width)
/Users/stabilini/Desktop/Benasque_hack/.venv/lib/python3.13/site-packages/tenpy/networks/mps.py:1629: UserWarning: unit_cell_width is a new argument for MPS and similar classes. It is optional for now, but will become mandatory in a future release. The default value (unit_cell_width=len(sites)) is correct, iff the lattice is a Chain. For other lattices, it is incorrect. It is used for dipolar charges and correlation_function2.
  super().__init__(sites, bc, unit_cell_width)
